In [ ]:
# ============================================================
# HCLA RQ2 FAST FULL VERSION
# - FULL experiment preserved: 100 cases, 2 conditions, 3 generations
# - Unused global SHAP removed
# - Memory cleanup added
# - OpenRouter generation/judging parallelized (5 workers)
# - Checkpoint/resume preserved
# ============================================================

import os, json, re, time, random, pickle, tempfile, warnings, glob, getpass, gc
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, average_precision_score, roc_auc_score,
    confusion_matrix, precision_recall_curve
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import shap
from openai import OpenAI

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)




In [ ]:
# ===== original cell 5 =====
class ResidualBlock(nn.Module):
    def __init__(self, d, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d, d), nn.BatchNorm1d(d),
        )
        self.act = nn.ReLU()
    def forward(self, x):
        return self.act(x + self.net(x))

class TabularResNetNet(nn.Module):
    def __init__(self, n_features, d=128, n_blocks=3, dropout=0.1):
        super().__init__()
        self.input = nn.Sequential(nn.Linear(n_features, d), nn.BatchNorm1d(d), nn.ReLU())
        self.blocks = nn.Sequential(*[ResidualBlock(d, dropout) for _ in range(n_blocks)])
        self.head = nn.Linear(d, 1)
    def forward(self, x):
        return self.head(self.blocks(self.input(x))).squeeze(1)

class NumericTokenizer(nn.Module):
    def __init__(self, n_features, d_token):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_features, d_token) * 0.02)
        self.bias = nn.Parameter(torch.zeros(n_features, d_token))
    def forward(self, x):
        return x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)

class FTTransformerNet(nn.Module):
    def __init__(self, n_features, d_token=64, n_heads=8, n_layers=3, dropout=0.1):
        super().__init__()
        self.tokenizer = NumericTokenizer(n_features, d_token)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_token))
        layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, dim_feedforward=d_token * 4,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_token)
        self.head = nn.Linear(d_token, 1)
    def forward(self, x):
        tok = self.tokenizer(x)
        cls = self.cls.expand(x.shape[0], -1, -1)
        z = self.encoder(torch.cat([cls, tok], dim=1))
        return self.head(self.norm(z[:, 0])).squeeze(1)

class TorchBinaryClassifier:
    def __init__(self, architecture, seed=42, epochs=20, batch_size=4096, patience=5):
        self.architecture = architecture
        self.seed = seed
        self.epochs = epochs
        self.batch_size = batch_size
        self.patience = patience
        self.scaler = StandardScaler()
        self.model = None
    def predict_proba(self, X):
        Xs = self.scaler.transform(X).astype(np.float32)
        loader = DataLoader(TensorDataset(torch.tensor(Xs, dtype=torch.float32)), batch_size=self.batch_size, shuffle=False)
        self.model = self.model.to(DEVICE)
        self.model.eval()
        probs = []
        with torch.no_grad():
            for (xb,) in loader:
                p = torch.sigmoid(self.model(xb.to(DEVICE))).cpu().numpy()
                probs.append(p)
        p1 = np.concatenate(probs)
        return np.column_stack([1 - p1, p1])




In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ===== original cell 7 =====

import os, glob

# ------------------------------------------------------------
# Main experiment settings: no changes required
# ------------------------------------------------------------
RQ2_EXPERIMENT = "E2_Random_AddressRemoved"
RQ2_MODEL_SEED = 42
RQ2_SPLIT_SEED = 42

N_NORMAL = 25
N_ANOMALY = 25
TOP_K_EVIDENCE = 5
N_GENERATIONS = 1

# RQ2 requirements
MAIN_CASES = 50               # Main unsupported-claim evaluation
MAIN_REPETITIONS = 1          # 50 cases × 1 repetition
STABILITY_CASES = 10          # Repeated-explanation stability evaluation
STABILITY_REPETITIONS = 10    # 10 cases × 10 repetitions
RQ2_PILOT = False

SHAP_BACKGROUND_N = 200

# Execution-speed settings: parallelize API requests without changing sample sizes or experimental conditions.
QWEN_WORKERS = 5
LLAMA_WORKERS = 5
CHECKPOINT_EVERY = 10   # Save after every ten completed jobs to reduce Drive I/O.

# ------------------------------------------------------------
# 1) Check the existing project's standard path first.
# ------------------------------------------------------------
DEFAULT_OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/Results/HCLA_MultiModel_Qwen_Llama_OpenRouter_SHAP"
DEFAULT_PKL = os.path.join(DEFAULT_OUTPUT_DIR, "rq1_multimodel_results_3seeds.pkl")
DEFAULT_DATA = "/content/drive/MyDrive/Colab Notebooks/Data/BTC/ETH MixingTornadoCash DF.csv"

def find_one(preferred, patterns, label):
    if os.path.exists(preferred):
        print(f"✅ {label} using default path")
        return preferred

    matches = []
    for pat in patterns:
        matches.extend(glob.glob(pat, recursive=True))
    matches = sorted(set(p for p in matches if os.path.isfile(p)))

    if not matches:
        raise FileNotFoundError(
            f"{label} was not found.\n"
            f"Default path: {preferred}\n"
            "Check that the file exists in Google Drive."
        )

    # Prefer the most recently modified file.
    matches.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    print(f"✅ {label} automatically found:", matches[0])
    if len(matches) > 1:
        print(f"   Selected the most recent of {len(matches)} candidate files")
    return matches[0]

RQ1_PKL_PATH = find_one(
    DEFAULT_PKL,
    [
        "/content/drive/MyDrive/**/rq1_multimodel_results_3seeds.pkl",
        "/content/drive/MyDrive/**/rq1*multimodel*3seeds*.pkl",
    ],
    "RQ1 PKL"
)

DATA_PATH = find_one(
    DEFAULT_DATA,
    [
        "/content/drive/MyDrive/**/ETH MixingTornadoCash DF.csv",
        "/content/drive/MyDrive/**/*TornadoCash*.csv",
    ],
    "Original CSV"
)

# Use the directory containing the PKL as the base OUTPUT_DIR for RQ2 results.
OUTPUT_DIR = os.path.dirname(RQ1_PKL_PATH)

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "_checkpoints")
RQ2_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "RQ2_Evidence_Groundedness")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RQ2_OUTPUT_DIR, exist_ok=True)

print("\n================ AUTO CONFIG ================")
print("DATA_PATH    :", DATA_PATH)
print("RQ1_PKL_PATH :", RQ1_PKL_PATH)
print("OUTPUT_DIR   :", OUTPUT_DIR)
print("RQ2_OUT      :", RQ2_OUTPUT_DIR)
print("MODE         :", "PILOT" if RQ2_PILOT else "FULL")
print("=============================================")




In [ ]:
# ===== original cell 8 =====
def atomic_pickle_dump(obj, path):
    fd, tmp = tempfile.mkstemp(prefix=".tmp_", dir=os.path.dirname(path))
    os.close(fd)
    try:
        with open(tmp, "wb") as f:
            pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp): os.remove(tmp)

def atomic_json_dump(obj, path):
    fd, tmp = tempfile.mkstemp(prefix=".tmp_", dir=os.path.dirname(path))
    os.close(fd)
    try:
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False, indent=2)
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp): os.remove(tmp)

def atomic_csv_save(df, path):
    fd, tmp = tempfile.mkstemp(prefix=".tmp_", suffix=".csv", dir=os.path.dirname(path))
    os.close(fd)
    try:
        df.to_csv(tmp, index=False, encoding="utf-8-sig")
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp): os.remove(tmp)

def load_csv_or_empty(path, columns=None):
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"✅ checkpoint loaded: {os.path.basename(path)} ({len(df):,} rows)")
        return df
    return pd.DataFrame(columns=columns)

def keyset(df, cols):
    if df is None or len(df) == 0 or any(c not in df.columns for c in cols):
        return set()
    return set(map(tuple, df[cols].astype(str).itertuples(index=False, name=None)))




In [ ]:
# ===== LIGHT RQ2: LightGBM only + exact 100-row CSV extraction =====
# Key points:
# 1) Retain only the LightGBM / E2 / seed42 run from the RQ1 PKL.
# 2) Do not load all original CSV features into memory.
# 3) Read only Date and label columns to reproduce the exact RQ1 70/30 split order.
# 4) Select 50 normal and 50 anomalous cases from the test set using seed 42.
# 5) Read full feature values only for the 100 selected original rows.
# 6) Apply the saved LightGBM model and preprocessing components, then compute SHAP for only 100 cases.

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"DATA_PATH not found: {DATA_PATH}")
if not os.path.exists(RQ1_PKL_PATH):
    raise FileNotFoundError(f"RQ1 PKL not found: {RQ1_PKL_PATH}")

DATE_COL = "Date"
LABEL_COL = "label"
ADDRESS_COLS = ["Receiving Address", "Counterparty Address"]
ALWAYS_DROP_COLS = [
    "Hash", "Source", "Counterparty Cluster Name",
    "Counterparty Shared Name", "Counterparty Category",
]
TIME_FEATURES = ["Year", "Month", "Day", "Hour", "DayOfWeek"]

SELECTED_MODEL = "LightGBM"

def get_result_run(results, model_name, experiment_name, seed):
    found = [
        r for r in results
        if r["model_name"] == model_name
        and r["name"] == experiment_name
        and r["metrics"]["seed"] == seed
    ]
    if len(found) != 1:
        raise ValueError(
            f"Expected one run, found {len(found)} for "
            f"{model_name}/{experiment_name}/seed={seed}"
        )
    return found[0]

def model_input_from_raw(run, X_df):
    arr = run["imputer"].transform(X_df)
    if run["scaler"] is not None:
        arr = run["scaler"].transform(arr)
    return arr

def add_time_features(frame):
    out = frame.copy()
    dt = pd.to_datetime(out[DATE_COL], errors="coerce")
    out["Year"] = dt.dt.year
    out["Month"] = dt.dt.month
    out["Day"] = dt.dt.day
    out["Hour"] = dt.dt.hour
    out["DayOfWeek"] = dt.dt.dayofweek
    return out

# ------------------------------------------------------------
# A. Load the PKL and retain only the LightGBM E2 seed42 run.
# ------------------------------------------------------------
print("Loading RQ1 PKL...")
_t0 = time.time()
with open(RQ1_PKL_PATH, "rb") as f:
    results = pickle.load(f)
print(f"✅ RQ1 runs loaded: {len(results)} ({time.time()-_t0:.1f}s)")

rq2_run = get_result_run(
    results,
    SELECTED_MODEL,
    RQ2_EXPERIMENT,
    RQ2_MODEL_SEED,
)
print("✅ Selected run:", rq2_run["model_name"], rq2_run["name"], rq2_run["metrics"]["seed"])

feature_names = list(rq2_run["feature_names"])
y_saved = np.asarray(rq2_run["y_test"]).astype(int)

# Retain one LightGBM run and immediately release the other 47 runs.
# rq2_run refers to an object within results and remains valid after deleting results.
del results
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ Other RQ1 runs released from memory")

# ------------------------------------------------------------
# B. Read only Date and label columns to reconstruct split indices.
#    (Never load the full feature dataset into memory.)
# ------------------------------------------------------------
print("Reading only Date + label to reproduce exact split...")
_t0 = time.time()
meta = pd.read_csv(DATA_PATH, usecols=[DATE_COL, LABEL_COL])
meta["_csv_row"] = np.arange(len(meta), dtype=np.int64)
meta[DATE_COL] = pd.to_datetime(meta[DATE_COL], errors="coerce")
meta = meta.dropna(subset=[DATE_COL, LABEL_COL]).copy()
meta[LABEL_COL] = meta[LABEL_COL].astype(int)
print(f"✅ Meta loaded: {len(meta):,} valid rows ({time.time()-_t0:.1f}s)")

# Apply the same row-level 70/30 stratified split as split_random(df, 42).
_, test_meta = train_test_split(
    meta,
    test_size=0.30,
    random_state=RQ2_SPLIT_SEED,
    stratify=meta[LABEL_COL],
)

y_rebuilt = test_meta[LABEL_COL].to_numpy(dtype=int)
if len(y_saved) != len(y_rebuilt) or not np.array_equal(y_saved, y_rebuilt):
    raise ValueError(
        "Rebuilt E2 y_test does not match RQ1 PKL. "
        "Check DATA_PATH, data version, and split seed."
    )
print(f"✅ Exact RQ1 test split reproduced: {len(y_rebuilt):,} rows")

# ------------------------------------------------------------
# C. Preserve the original conditions: 50 Normal + 50 Anomaly from the test set, seed 42.
# ------------------------------------------------------------
rng = np.random.default_rng(RQ2_MODEL_SEED)
normal_pos = np.where(y_saved == 0)[0]
anomaly_pos = np.where(y_saved == 1)[0]

if len(normal_pos) < N_NORMAL or len(anomaly_pos) < N_ANOMALY:
    raise ValueError(
        f"Not enough test samples: normal={len(normal_pos)}, anomaly={len(anomaly_pos)}"
    )

sample_normal = rng.choice(normal_pos, size=N_NORMAL, replace=False)
sample_anomaly = rng.choice(anomaly_pos, size=N_ANOMALY, replace=False)
sample_idx = np.concatenate([sample_normal, sample_anomaly])
rng.shuffle(sample_idx)

# sample_idx = position within the RQ1 test set (same as the original case_id).
# selected_csv_rows = actual zero-based row indices in the original CSV.
selected_csv_rows = test_meta.iloc[sample_idx]["_csv_row"].to_numpy(dtype=np.int64)

print("✅ RQ2 sample fixed:", len(sample_idx), "cases")
print("   Normal:", int((y_saved[sample_idx] == 0).sum()))
print("   Anomaly:", int((y_saved[sample_idx] == 1).sum()))

# Release meta/test_meta immediately after obtaining split indices.
selected_csv_rows_ordered = selected_csv_rows.copy()
del meta, test_meta, y_rebuilt, normal_pos, anomaly_pos, sample_normal, sample_anomaly
gc.collect()

# ------------------------------------------------------------
# D. Read full feature data from only the 100 selected CSV rows.
# ------------------------------------------------------------
# Recreate time-derived features from Date; do not read them from the source CSV.
source_feature_cols = [c for c in feature_names if c not in TIME_FEATURES]
needed_cols = set([DATE_COL, LABEL_COL] + source_feature_cols)

# Check the header and pass only existing columns to usecols.
header_cols = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
missing_source = [c for c in source_feature_cols if c not in header_cols]
if missing_source:
    raise ValueError(f"Required source columns missing from CSV: {missing_source[:20]}")
usecols = [c for c in header_cols if c in needed_cols]

selected_set = set(map(int, selected_csv_rows_ordered))

def _skip_unselected_csv_rows(line_no):
    # line_no=0 is the header; data row n corresponds to line_no=n+1.
    return line_no > 0 and (line_no - 1) not in selected_set

print("Reading ONLY the selected 100 feature rows...")
_t0 = time.time()
selected_df = pd.read_csv(
    DATA_PATH,
    usecols=usecols,
    skiprows=_skip_unselected_csv_rows,
)

if len(selected_df) != len(selected_csv_rows_ordered):
    raise ValueError(
        f"Expected {len(selected_csv_rows_ordered)} selected rows, got {len(selected_df)}"
    )

# skiprows preserves file order; attach original row IDs and restore sample_idx ordering.
selected_df["_csv_row"] = np.sort(selected_csv_rows_ordered)
selected_df = (
    selected_df.set_index("_csv_row")
    .loc[selected_csv_rows_ordered]
    .reset_index()
)
print(f"✅ 100 feature rows loaded ({time.time()-_t0:.1f}s)")

# ------------------------------------------------------------
# E. Preprocess only 100 cases, run LightGBM, then compute SHAP.
# ------------------------------------------------------------
selected_feat = add_time_features(selected_df)
selected_labels = selected_feat[LABEL_COL].astype(int).to_numpy()

# Verify that the selected original labels match the RQ1 test labels exactly.
if not np.array_equal(selected_labels, y_saved[sample_idx]):
    raise ValueError("Selected 100 labels do not match saved RQ1 test labels.")

X_selected_raw = selected_feat.drop(
    columns=[LABEL_COL, DATE_COL, "_csv_row"] + ALWAYS_DROP_COLS + ADDRESS_COLS,
    errors="ignore",
).copy()

# Match original preprocessing: drop object/category columns and enforce the RQ1 feature order.
obj_cols = X_selected_raw.select_dtypes(include=["object", "category"]).columns.tolist()
if obj_cols:
    X_selected_raw = X_selected_raw.drop(columns=obj_cols, errors="ignore")

missing_features = [c for c in feature_names if c not in X_selected_raw.columns]
if missing_features:
    raise ValueError(f"Missing reconstructed features: {missing_features}")

X_selected_raw = X_selected_raw.reindex(columns=feature_names)
X_sample = model_input_from_raw(rq2_run, X_selected_raw)

# Check consistency between saved predictions/probabilities and current LightGBM inference.
pred_now = np.asarray(rq2_run["model"].predict(X_sample)).astype(int)
prob_now = np.asarray(rq2_run["model"].predict_proba(X_sample))[:, 1]
pred_saved = np.asarray(rq2_run["y_pred"])[sample_idx].astype(int)
prob_saved = np.asarray(rq2_run["y_prob"])[sample_idx].astype(float)

if not np.array_equal(pred_now, pred_saved):
    raise ValueError("LightGBM predictions differ from saved RQ1 predictions.")
if not np.allclose(prob_now, prob_saved, rtol=1e-6, atol=1e-8):
    print("⚠️ Probability values show tiny numeric differences; saved RQ1 probabilities will be used.")

print("Running SHAP for ONLY 100 cases...")
_t0 = time.time()
shap_explainer = shap.TreeExplainer(rq2_run["model"])
raw_sample = shap_explainer(X_sample)
sample_shap = raw_sample.values[:, :, 1] if raw_sample.values.ndim == 3 else raw_sample.values
print(f"✅ SHAP complete: {sample_shap.shape} ({time.time()-_t0:.1f}s)")

# ------------------------------------------------------------
# F. Create evidence packages using the original RQ2 format and conditions.
# ------------------------------------------------------------
def safe_scalar(v):
    if pd.isna(v):
        return None
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, np.floating):
        return float(v)
    return v

def build_evidence_package(local_pos, test_pos, top_k=5):
    row = X_selected_raw.iloc[local_pos]
    sv = sample_shap[local_pos]
    order = np.argsort(np.abs(sv))[::-1][:top_k]

    top_evidence = []
    for rank, j in enumerate(order, 1):
        top_evidence.append({
            "rank": rank,
            "feature": feature_names[j],
            "value": safe_scalar(row.iloc[j]),
            "shap_value": float(sv[j]),
            "effect": "toward_anomaly" if sv[j] > 0 else "toward_normal",
        })

    return {
        # Original case_id refers to test-set position, not a CSV row; preserve it.
        "case_id": int(test_pos),
        "detector": SELECTED_MODEL,
        "true_label": "Anomaly" if int(y_saved[test_pos]) == 1 else "Normal",
        "prediction": "Anomaly" if int(rq2_run["y_pred"][test_pos]) == 1 else "Normal",
        "anomaly_probability": float(rq2_run["y_prob"][test_pos]),
        "top_evidence": top_evidence,
    }

evidence_packages = [
    build_evidence_package(local_pos=i, test_pos=int(test_pos), top_k=TOP_K_EVIDENCE)
    for i, test_pos in enumerate(sample_idx)
]

EVIDENCE_CKPT = os.path.join(CHECKPOINT_DIR, "evidence_packages.pkl")
atomic_pickle_dump(evidence_packages, EVIDENCE_CKPT)
print("✅ Evidence packages:", len(evidence_packages))
print("Saved:", EVIDENCE_CKPT)
print(json.dumps(evidence_packages[0], ensure_ascii=False, indent=2))

# Release unnecessary large objects before API calls.
# rq2_run is no longer needed for Qwen/Llama after generating evidence.
del selected_df, selected_feat, X_selected_raw, X_sample, raw_sample, sample_shap
del pred_now, prob_now, pred_saved, prob_saved, selected_csv_rows_ordered, selected_set
# rq2_run, including its model object, can be released after evidence generation.
del rq2_run
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ LightGBM/SHAP objects released. API stage can start with low RAM.")




In [ ]:
# ===== original cell 18 =====

import os

def load_openrouter_key():
    """Read OpenRouter credentials without embedding them in this notebook."""
    key = (os.environ.get("OPENROUTER_API_KEY") or "").strip()
    if not key:
        try:
            from google.colab import userdata
            key = (userdata.get("OPENROUTER_API_KEY") or "").strip()
        except (ImportError, KeyError, AttributeError):
            pass
    if not key:
        raise RuntimeError(
            "Set OPENROUTER_API_KEY as an environment variable or a Colab Secret before running."
        )
    return key

OPENROUTER_API_KEY = load_openrouter_key()

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
QWEN_MODEL = "qwen/qwen3-235b-a22b-2507"
LLAMA_JUDGE_MODEL = "meta-llama/llama-3.3-70b-instruct"
GEN_TEMPERATURE = 0.2
JUDGE_TEMPERATURE = 0.0

openrouter_client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    timeout=120.0,
    max_retries=0  # Use only the custom retry/backoff below to avoid duplicate retries.
)

def qwen_generate(prompt, temperature=GEN_TEMPERATURE, max_retries=5):
    for attempt in range(max_retries):
        try:
            r = openrouter_client.chat.completions.create(
                model=QWEN_MODEL,
                messages=[{"role":"user","content":prompt}],
                temperature=temperature
            )
            text = r.choices[0].message.content
            if text is None:
                raise RuntimeError("Qwen returned empty output")
            return text.strip()
        except Exception as e:
            print(f"Qwen/OpenRouter error ({attempt+1}/{max_retries}): {e}")
            if attempt >= max_retries - 1:
                raise
            time.sleep(min(60, 5 * (2 ** attempt)))

def llama_judge(prompt, temperature=JUDGE_TEMPERATURE, max_retries=6):
    for attempt in range(max_retries):
        try:
            r = openrouter_client.chat.completions.create(
                model=LLAMA_JUDGE_MODEL,
                messages=[{"role":"user","content":prompt}],
                temperature=temperature
            )
            text = r.choices[0].message.content
            if text is None:
                raise RuntimeError("Llama returned empty output")
            return text.strip()
        except Exception as e:
            print(f"Llama/OpenRouter error ({attempt+1}/{max_retries}): {e}")
            if attempt >= max_retries - 1:
                raise
            time.sleep(min(60, 5 * (2 ** attempt)))

# Check API connectivity in advance.
print("Qwen Generator:", QWEN_MODEL)
print("Llama Judge   :", LLAMA_JUDGE_MODEL)
print("Testing OpenRouter connection...")
print("Qwen :", qwen_generate("Reply with exactly: QWEN OK", temperature=0.0))
print("Llama:", llama_judge("Reply with exactly: LLAMA OK", temperature=0.0))




In [ ]:
# ===== original cell 20 =====
HCLA_SYSTEM = """
You are the Justification component of HCLA for transaction anomaly analysis.
Generate a concise explanation using ONLY the supplied evidence.
Do not introduce external facts, address history, criminal attribution,
causal claims, or domain facts not explicitly present in the evidence package.
If evidence is insufficient, state that limitation.
Explain the evidence in plain language.
""".strip()

BASELINE_SYSTEM = """
You are an AI assistant explaining a transaction anomaly prediction.
Use the supplied transaction information to provide a concise and useful explanation
for why the model may have produced this prediction.
""".strip()

def make_narrative_prompt(evidence, condition):
    system = HCLA_SYSTEM if condition == "HCLA" else BASELINE_SYSTEM
    return f"""{system}

Evidence package:
{json.dumps(evidence, ensure_ascii=False, indent=2)}

Return one short paragraph (2-4 sentences).
"""

def generate_narratives(evidence_list, n_generations=3, checkpoint_path=None):
    checkpoint_path = checkpoint_path or os.path.join(CHECKPOINT_DIR, "rq2_narratives.csv")
    cols = ["case_id","true_label","prediction","condition","rep","narrative","evidence_json"]
    df0 = load_csv_or_empty(checkpoint_path, cols)
    rows = df0.to_dict("records") if len(df0) else []
    done = keyset(df0, ["case_id","condition","rep"])
    total = len(evidence_list) * 2 * n_generations

    tasks = []
    for e in evidence_list:
        for condition in ["Baseline", "HCLA"]:
            for rep in range(1, n_generations + 1):
                key = (str(e["case_id"]), condition, str(rep))
                if key not in done:
                    tasks.append((e, condition, rep, key))

    print(f"RQ2 Narrative resume {len(done)}/{total}")
    print(f"🚀 Qwen parallel workers: {QWEN_WORKERS} / remaining: {len(tasks)}")
    if not tasks:
        return pd.DataFrame(rows, columns=cols)

    completed_since_save = 0
    with ThreadPoolExecutor(max_workers=QWEN_WORKERS) as executor:
        future_map = {
            executor.submit(qwen_generate, make_narrative_prompt(e, condition)): (e, condition, rep, key)
            for e, condition, rep, key in tasks
        }

        for future in as_completed(future_map):
            e, condition, rep, key = future_map[future]
            text = future.result()
            rows.append({
                "case_id": e["case_id"],
                "true_label": e["true_label"],
                "prediction": e["prediction"],
                "condition": condition,
                "rep": rep,
                "narrative": text,
                "evidence_json": json.dumps(e, ensure_ascii=False),
            })
            done.add(key)
            completed_since_save += 1

            if completed_since_save >= CHECKPOINT_EVERY or len(done) == total:
                atomic_csv_save(pd.DataFrame(rows, columns=cols), checkpoint_path)
                completed_since_save = 0

            print(f"✅ RQ2 Narrative {len(done)}/{total}")

    out = pd.DataFrame(rows, columns=cols)
    out = out.sort_values(["case_id", "condition", "rep"]).reset_index(drop=True)
    atomic_csv_save(out, checkpoint_path)
    return out




In [ ]:
# ===== original cell 22 =====
def extract_json_object(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text, flags=re.I).strip()
    text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.S)
        if not m: raise ValueError(text[:1000])
        return json.loads(m.group(0))

GROUNDING_JUDGE = """
You are a strict evidence-grounding evaluator.
Evaluate the explanation ONLY against the supplied evidence package.

Split the explanation into atomic factual claims.
For each claim, label it:
- SUPPORTED
- UNSUPPORTED
- CONTRADICTED

Also list which supplied top-evidence feature names were actually used.

Return ONLY JSON:
{
  "claims": [
    {"claim":"...", "label":"SUPPORTED|UNSUPPORTED|CONTRADICTED", "reason":"..."}
  ],
  "used_evidence_features": ["..."]
}
""".strip()

def judge_narrative(evidence, narrative):
    prompt = f"""{GROUNDING_JUDGE}

Evidence package:
{json.dumps(evidence, ensure_ascii=False, indent=2)}

Explanation:
{narrative}
"""
    return extract_json_object(llama_judge(prompt, temperature=0.0))

def evaluate_narratives(narratives_df, checkpoint_path=None, progress_path=None):
    checkpoint_path = checkpoint_path or os.path.join(CHECKPOINT_DIR, "rq2_claims.csv")
    progress_path = progress_path or os.path.join(CHECKPOINT_DIR, "rq2_judge_done.json")
    cols = ["case_id","condition","rep","claim","label","reason","used_evidence_features","evidence_json","narrative"]
    df0 = load_csv_or_empty(checkpoint_path, cols)
    rows = df0.to_dict("records") if len(df0) else []

    if os.path.exists(progress_path):
        with open(progress_path, "r", encoding="utf-8") as f:
            done = {tuple(map(str, x)) for x in json.load(f)}
    else:
        done = set()

    total = len(narratives_df)
    tasks = []
    for _, row in narratives_df.iterrows():
        key = (str(row["case_id"]), str(row["condition"]), str(row["rep"]))
        if key not in done:
            tasks.append((row.to_dict(), key))

    print(f"RQ2 Judge resume {len(done)}/{total}")
    print(f"🚀 Llama parallel workers: {LLAMA_WORKERS} / remaining: {len(tasks)}")
    if not tasks:
        return pd.DataFrame(rows, columns=cols)

    completed_since_save = 0
    with ThreadPoolExecutor(max_workers=LLAMA_WORKERS) as executor:
        future_map = {}
        for row, key in tasks:
            evidence = json.loads(row["evidence_json"])
            future = executor.submit(judge_narrative, evidence, row["narrative"])
            future_map[future] = (row, key)

        for future in as_completed(future_map):
            row, key = future_map[future]
            judged = future.result()
            used = judged.get("used_evidence_features", [])
            claims = judged.get("claims", []) or [{
                "claim": row["narrative"],
                "label": "UNSUPPORTED",
                "reason": "No claims parsed"
            }]

            for x in claims:
                rows.append({
                    "case_id": row["case_id"],
                    "condition": row["condition"],
                    "rep": row["rep"],
                    "claim": x.get("claim", ""),
                    "label": str(x.get("label", "")).upper(),
                    "reason": x.get("reason", ""),
                    "used_evidence_features": json.dumps(used, ensure_ascii=False),
                    "evidence_json": row["evidence_json"],
                    "narrative": row["narrative"],
                })

            done.add(key)
            completed_since_save += 1

            if completed_since_save >= CHECKPOINT_EVERY or len(done) == total:
                atomic_csv_save(pd.DataFrame(rows, columns=cols), checkpoint_path)
                atomic_json_dump(sorted([list(x) for x in done]), progress_path)
                completed_since_save = 0

            print(f"✅ RQ2 Judge {len(done)}/{total}")

    out = pd.DataFrame(rows, columns=cols)
    out = out.sort_values(["case_id", "condition", "rep", "claim"]).reset_index(drop=True)
    atomic_csv_save(out, checkpoint_path)
    atomic_json_dump(sorted([list(x) for x in done]), progress_path)
    return out


def summarize_groundedness(claims_df):
    rows = []
    for key, g in claims_df.groupby(["case_id","condition","rep"]):
        labels = g["label"].tolist(); n = len(labels)
        evidence = json.loads(g.iloc[0]["evidence_json"])
        supplied = {x["feature"] for x in evidence["top_evidence"]}
        used = set()
        for s in g["used_evidence_features"]:
            try: used.update(json.loads(s))
            except Exception: pass
        rows.append({
            "case_id":key[0], "condition":key[1], "rep":key[2],
            "ESR":labels.count("SUPPORTED")/n if n else np.nan,
            "UCR":labels.count("UNSUPPORTED")/n if n else np.nan,
            "CR":labels.count("CONTRADICTED")/n if n else np.nan,
            "EC":len(supplied.intersection(used))/len(supplied) if supplied else np.nan,
        })
    metric_df = pd.DataFrame(rows)
    mean_df = metric_df.groupby("condition")[["ESR","UCR","CR","EC"]].mean()
    sd_df = metric_df.groupby("condition")[["ESR","UCR","CR","EC"]].std(ddof=1)
    paper = pd.DataFrame(index=mean_df.index)
    for metric in ["ESR","UCR","CR","EC"]:
        paper[metric] = [f"{mean_df.loc[x,metric]:.4f} ± {sd_df.loc[x,metric]:.4f}" for x in mean_df.index]
    return metric_df, mean_df, sd_df, paper.reset_index()




In [ ]:
# ============================================================
# RQ2 Revised Experiment
# 50 cases × 1 + 10 cases × 10
#
# 1) Unsupported Claim:
#    50 cases × 2 conditions × 1 rep = 100 explanations
#
# 2) Stability:
#    10 cases × 2 conditions × 10 reps = 200 explanations
#
# TOTAL
# Qwen       = 300 calls
# Llama Judge= 300 calls
# API calls  = 600
#
# Additional:
# - Measure execution time
# - Aggregate input / output / total tokens
# - Aggregate API cost (if the column exists)
# - Save a usage-summary CSV
# ============================================================

import os
import time
import json
import re
import pandas as pd
import numpy as np


# ============================================================
# JSON SAFE PARSER
# Prevent a malformed Llama judge JSON response from terminating the full experiment.
# ============================================================

def extract_json_object(text):
    raw_text = "" if text is None else str(text)
    text = raw_text.strip()

    # Remove Markdown code fences.
    text = re.sub(r"^\s*```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```\s*$", "", text).strip()

    # 1) Parse valid JSON.
    try:
        result = json.loads(text)
        if isinstance(result, dict):
            result.setdefault("_parse_error", False)
            return result
    except Exception:
        pass

    # 2) Extract content from the first { to the last }.
    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1 or end <= start:
        print("⚠️ JSON parse failed: JSON object not found")
        return {
            "claims": [],
            "used_evidence_features": [],
            "_parse_error": True,
            "_parse_error_reason": "JSON object not found",
            "_raw_response": raw_text[:4000]
        }

    candidate = text[start:end + 1]

    # 3) Parse the extracted JSON as is.
    try:
        result = json.loads(candidate)
        if isinstance(result, dict):
            result.setdefault("_parse_error", False)
            return result
    except Exception:
        pass

    # 4) Repair common LLM-generated JSON errors.
    repaired = candidate
    repaired = repaired.replace("\ufeff", "")
    repaired = (
        repaired
        .replace("“", '"')
        .replace("”", '"')
        .replace("‘", "'")
        .replace("’", "'")
    )

    # trailing comma
    repaired = re.sub(r",\s*([}\]])", r"\1", repaired)

    # Missing comma between objects.
    repaired = re.sub(r"}\s*{", "},{", repaired)

    # Missing comma before the next key after a newline.
    repaired = re.sub(
        r'("|\}|\])(\s*\n\s*)("(?=[A-Za-z_][^"\n]{0,100}"\s*:))',
        r'\1,\2\3',
        repaired
    )

    repaired = repaired.replace("\t", " ").replace("\r", " ")

    # 5) Parse the repaired JSON.
    try:
        result = json.loads(repaired, strict=False)
        if isinstance(result, dict):
            result["_parse_error"] = False
            result["_json_repaired"] = True
            return result

    except Exception as e:
        print()
        print("⚠️ JSON PARSE FAILED - SKIPPING THIS JUDGE RESPONSE")
        print("Reason:", e)

        # Do not raise an exception; continue the full experiment.
        return {
            "claims": [],
            "used_evidence_features": [],
            "_parse_error": True,
            "_parse_error_reason": str(e),
            "_raw_response": raw_text[:4000]
        }


print("✅ JSON-safe parser loaded")


# ============================================================
# 0. Helper Functions
# ============================================================

def normalize_label(label):
    """
    true_label:
      0 / 1
      "0" / "1"
      "Normal" / "Anomaly"
    All formats are supported.
    """
    s = str(label).strip().lower()

    if s in ["0", "normal"]:
        return 0

    elif s in ["1", "anomaly"]:
        return 1

    else:
        raise ValueError(
            f"Unknown true_label value: {label}"
        )


# ------------------------------------------------------------
# Automatically detect token / cost columns.
# ------------------------------------------------------------
def summarize_api_usage(df, stage_name):
    """
    Automatically find and sum available token and cost columns in the DataFrame.

    Examples supported:
      input_tokens
      prompt_tokens

      output_tokens
      completion_tokens

      total_tokens

      cost
      cost_usd
      api_cost
      api_cost_usd

      latency
      latency_sec
      elapsed_sec
    """

    if df is None or len(df) == 0:
        return {
            "stage": stage_name,
            "rows": 0,
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "cost_usd": np.nan,
            "avg_latency_sec": np.nan
        }

    # -------------------------
    # Candidate columns
    # -------------------------

    input_candidates = [
        "input_tokens",
        "prompt_tokens",
        "usage_prompt_tokens",
        "usage_input_tokens"
    ]

    output_candidates = [
        "output_tokens",
        "completion_tokens",
        "usage_completion_tokens",
        "usage_output_tokens"
    ]

    total_candidates = [
        "total_tokens",
        "usage_total_tokens"
    ]

    cost_candidates = [
        "cost_usd",
        "api_cost_usd",
        "cost",
        "api_cost",
        "usage_cost"
    ]

    latency_candidates = [
        "latency_sec",
        "latency",
        "elapsed_sec",
        "response_time_sec"
    ]


    def find_column(candidates):
        for col in candidates:
            if col in df.columns:
                return col
        return None


    input_col = find_column(input_candidates)
    output_col = find_column(output_candidates)
    total_col = find_column(total_candidates)
    cost_col = find_column(cost_candidates)
    latency_col = find_column(latency_candidates)


    # -------------------------
    # Aggregate results
    # -------------------------

    input_tokens = 0
    output_tokens = 0
    total_tokens = 0
    cost_usd = np.nan
    avg_latency = np.nan


    if input_col:
        input_tokens = pd.to_numeric(
            df[input_col],
            errors="coerce"
        ).fillna(0).sum()


    if output_col:
        output_tokens = pd.to_numeric(
            df[output_col],
            errors="coerce"
        ).fillna(0).sum()


    if total_col:
        total_tokens = pd.to_numeric(
            df[total_col],
            errors="coerce"
        ).fillna(0).sum()

    elif input_col or output_col:
        total_tokens = input_tokens + output_tokens


    if cost_col:
        cost_usd = pd.to_numeric(
            df[cost_col],
            errors="coerce"
        ).fillna(0).sum()


    if latency_col:
        avg_latency = pd.to_numeric(
            df[latency_col],
            errors="coerce"
        ).mean()


    return {
        "stage": stage_name,
        "rows": len(df),

        "input_tokens": int(input_tokens),
        "output_tokens": int(output_tokens),
        "total_tokens": int(total_tokens),

        "cost_usd": (
            float(cost_usd)
            if not pd.isna(cost_usd)
            else np.nan
        ),

        "avg_latency_sec": (
            float(avg_latency)
            if not pd.isna(avg_latency)
            else np.nan
        ),

        "input_token_column": input_col,
        "output_token_column": output_col,
        "total_token_column": total_col,
        "cost_column": cost_col,
        "latency_column": latency_col
    }


# ============================================================
# 1. Main Evidence
# ============================================================

# evidence_packages =
# Normal 25 + Anomaly 25
main_evidence = evidence_packages[:MAIN_CASES]


# ============================================================
# 2. Stability Evidence
#    Normal 5 + Anomaly 5
# ============================================================

_stab_normal = [
    e for e in main_evidence
    if normalize_label(e["true_label"]) == 0
][:5]


_stab_anomaly = [
    e for e in main_evidence
    if normalize_label(e["true_label"]) == 1
][:5]


stability_evidence = (
    _stab_normal
    +
    _stab_anomaly
)


# ============================================================
# 3. Data Validation
# ============================================================

if len(main_evidence) != MAIN_CASES:

    raise RuntimeError(
        f"Main evidence must be {MAIN_CASES}, "
        f"got {len(main_evidence)}"
    )


if len(stability_evidence) != STABILITY_CASES:

    raise RuntimeError(
        f"Stability evidence must be "
        f"{STABILITY_CASES}, "
        f"got {len(stability_evidence)} "
        f"(Normal={len(_stab_normal)}, "
        f"Anomaly={len(_stab_anomaly)})"
    )


# ------------------------------------------------------------
# Label distribution
# ------------------------------------------------------------

print("Main label distribution:")

print({
    "Normal":
        sum(
            normalize_label(e["true_label"]) == 0
            for e in main_evidence
        ),

    "Anomaly":
        sum(
            normalize_label(e["true_label"]) == 1
            for e in main_evidence
        )
})


print()

print("Stability label distribution:")

print({
    "Normal": len(_stab_normal),
    "Anomaly": len(_stab_anomaly)
})


# ============================================================
# 4. Experiment Plan
# ============================================================

print()

print("=" * 70)
print("RQ2 EXPERIMENT PLAN")
print("=" * 70)


print(
    f"Main Unsupported Claim : "
    f"{len(main_evidence)} cases × "
    f"2 conditions × "
    f"{MAIN_REPETITIONS} rep"
)


print(
    f"Stability              : "
    f"{len(stability_evidence)} cases × "
    f"2 conditions × "
    f"{STABILITY_REPETITIONS} reps"
)


main_exp = (
    len(main_evidence)
    * 2
    * MAIN_REPETITIONS
)


stab_exp = (
    len(stability_evidence)
    * 2
    * STABILITY_REPETITIONS
)


print(
    f"Qwen generations       : "
    f"{main_exp} + {stab_exp} "
    f"= {main_exp + stab_exp}"
)


print(
    f"Llama judge calls      : "
    f"{main_exp} + {stab_exp} "
    f"= {main_exp + stab_exp}"
)


print(
    f"Total API calls        : "
    f"{(main_exp + stab_exp) * 2}"
)


print(
    f"Parallelism            : "
    f"Qwen={QWEN_WORKERS}, "
    f"Llama={LLAMA_WORKERS}"
)


print("=" * 70)


# ============================================================
# Full experiment start time
# ============================================================

_t_total = time.time()


# ============================================================
# A. MAIN
# 50 × 2 × 1
# ============================================================

MAIN_NARRATIVE_CKPT = os.path.join(
    CHECKPOINT_DIR,
    "rq2_main50_narratives.csv"
)


MAIN_CLAIMS_CKPT = os.path.join(
    CHECKPOINT_DIR,
    "rq2_main50_claims.csv"
)


MAIN_JUDGE_DONE = os.path.join(
    CHECKPOINT_DIR,
    "rq2_main50_judge_done.json"
)


# ------------------------------------------------------------
# A-1. Qwen Generation
# ------------------------------------------------------------

print()
print("=" * 70)
print("A-1. MAIN QWEN GENERATION")
print("=" * 70)

_t_main_qwen = time.time()


main_narratives_df = generate_narratives(
    main_evidence,
    n_generations=MAIN_REPETITIONS,
    checkpoint_path=MAIN_NARRATIVE_CKPT
)


main_qwen_sec = (
    time.time()
    -
    _t_main_qwen
)


print(
    f"✅ MAIN QWEN COMPLETE: "
    f"{main_qwen_sec / 60:.2f} min"
)


# ------------------------------------------------------------
# A-2. Llama Judge
# ------------------------------------------------------------

print()
print("=" * 70)
print("A-2. MAIN LLAMA JUDGE")
print("=" * 70)

_t_main_llama = time.time()


main_claims_df = evaluate_narratives(
    main_narratives_df,
    checkpoint_path=MAIN_CLAIMS_CKPT,
    progress_path=MAIN_JUDGE_DONE
)


main_llama_sec = (
    time.time()
    -
    _t_main_llama
)


print(
    f"✅ MAIN LLAMA COMPLETE: "
    f"{main_llama_sec / 60:.2f} min"
)


# ------------------------------------------------------------
# A-3. Groundedness Summary
# ------------------------------------------------------------

(
    main_metric_df,
    main_mean_df,
    main_sd_df,
    main_paper_df

) = summarize_groundedness(
    main_claims_df
)


# ------------------------------------------------------------
# A-4. Save
# ------------------------------------------------------------

atomic_csv_save(
    main_narratives_df,
    os.path.join(
        RQ2_OUTPUT_DIR,
        "rq2_main50_narratives.csv"
    )
)


atomic_csv_save(
    main_claims_df,
    os.path.join(
        RQ2_OUTPUT_DIR,
        "rq2_main50_claims.csv"
    )
)


atomic_csv_save(
    main_metric_df,
    os.path.join(
        RQ2_OUTPUT_DIR,
        "rq2_main50_metrics_by_case.csv"
    )
)


atomic_csv_save(
    main_paper_df,
    os.path.join(
        RQ2_OUTPUT_DIR,
        "rq2_main50_paper_summary.csv"
    )
)


main_total_sec = (
    main_qwen_sec
    +
    main_llama_sec
)


print()

print(
    f"✅ MAIN 50 COMPLETE "
    f"({main_total_sec / 60:.2f} min)"
)


display(main_paper_df)


# ============================================================
# B. STABILITY
# 10 × 2 × 10
# ============================================================

STAB_NARRATIVE_CKPT = os.path.join(
    CHECKPOINT_DIR,
    "rq2_stability10x10_narratives.csv"
)


STAB_CLAIMS_CKPT = os.path.join(
    CHECKPOINT_DIR,
    "rq2_stability10x10_claims.csv"
)


STAB_JUDGE_DONE = os.path.join(
    CHECKPOINT_DIR,
    "rq2_stability10x10_judge_done.json"
)


# ------------------------------------------------------------
# B-1. Qwen Generation
# ------------------------------------------------------------

print()
print("=" * 70)
print("B-1. STABILITY QWEN GENERATION")
print("=" * 70)


_t_stab_qwen = time.time()


stability_narratives_df = generate_narratives(
    stability_evidence,
    n_generations=STABILITY_REPETITIONS,
    checkpoint_path=STAB_NARRATIVE_CKPT
)


stab_qwen_sec = (
    time.time()
    -
    _t_stab_qwen
)


print(
    f"✅ STABILITY QWEN COMPLETE: "
    f"{stab_qwen_sec / 60:.2f} min"
)


# ------------------------------------------------------------
# B-2. Llama Judge
# ------------------------------------------------------------

print()
print("=" * 70)
print("B-2. STABILITY LLAMA JUDGE")
print("=" * 70)


_t_stab_llama = time.time()


stability_claims_df = evaluate_narratives(
    stability_narratives_df,
    checkpoint_path=STAB_CLAIMS_CKPT,
    progress_path=STAB_JUDGE_DONE
)


stab_llama_sec = (
    time.time()
    -
    _t_stab_llama
)


print(
    f"✅ STABILITY LLAMA COMPLETE: "
    f"{stab_llama_sec / 60:.2f} min"
)


# ------------------------------------------------------------
# B-3. Groundedness Summary
# ------------------------------------------------------------

(
    stability_metric_df,
    stability_mean_df,
    stability_sd_df,
    stability_paper_df

) = summarize_groundedness(
    stability_claims_df
)


# ============================================================
# Stability Statistics
# ============================================================

stability_by_rep_df = (

    stability_metric_df

    .groupby(
        ["condition", "rep"]
    )[
        ["ESR", "UCR", "CR", "EC"]
    ]

    .mean()

    .reset_index()
)


stability_overall_df = (

    stability_metric_df

    .groupby(
        "condition"
    )[
        ["ESR", "UCR", "CR", "EC"]
    ]

    .agg([
        "mean",
        "std"
    ])
)


# ============================================================
# Save Stability
# ============================================================

atomic_csv_save(
    stability_narratives_df,
    os.path.join(
        RQ2_OUTPUT_DIR,
        "rq2_stability10x10_narratives.csv"
    )
)


atomic_csv_save(
    stability_claims_df,
    os.path.join(
        RQ2_OUTPUT_DIR,
        "rq2_stability10x10_claims.csv"
    )
)


atomic_csv_save(
    stability_metric_df,
    os.path.join(
        RQ2_OUTPUT_DIR,
        "rq2_stability10x10_metrics.csv"
    )
)


atomic_csv_save(
    stability_by_rep_df,
    os.path.join(
        RQ2_OUTPUT_DIR,
        "rq2_stability10x10_by_rep.csv"
    )
)


stability_overall_df.to_csv(

    os.path.join(
        RQ2_OUTPUT_DIR,
        "rq2_stability10x10_overall.csv"
    ),

    encoding="utf-8-sig"
)


atomic_csv_save(
    stability_paper_df,

    os.path.join(
        RQ2_OUTPUT_DIR,
        "rq2_stability10x10_paper_summary.csv"
    )
)


stab_total_sec = (
    stab_qwen_sec
    +
    stab_llama_sec
)


print()

print(
    f"✅ STABILITY 10×10 COMPLETE "
    f"({stab_total_sec / 60:.2f} min)"
)


display(stability_paper_df)
display(stability_by_rep_df)


# ============================================================
# C. API TOKEN / COST SUMMARY
# ============================================================

print()
print("=" * 70)
print("RQ2 API TOKEN / COST SUMMARY")
print("=" * 70)


usage_rows = []


# Main Qwen
u = summarize_api_usage(
    main_narratives_df,
    "MAIN_QWEN"
)

u["elapsed_sec"] = main_qwen_sec

usage_rows.append(u)


# Main Llama
u = summarize_api_usage(
    main_claims_df,
    "MAIN_LLAMA_JUDGE"
)

u["elapsed_sec"] = main_llama_sec

usage_rows.append(u)


# Stability Qwen
u = summarize_api_usage(
    stability_narratives_df,
    "STABILITY_QWEN"
)

u["elapsed_sec"] = stab_qwen_sec

usage_rows.append(u)


# Stability Llama
u = summarize_api_usage(
    stability_claims_df,
    "STABILITY_LLAMA_JUDGE"
)

u["elapsed_sec"] = stab_llama_sec

usage_rows.append(u)


usage_df = pd.DataFrame(
    usage_rows
)


# ============================================================
# D. TOTAL
# ============================================================

total_elapsed_sec = (
    time.time()
    -
    _t_total
)


total_input_tokens = (
    usage_df["input_tokens"]
    .fillna(0)
    .sum()
)


total_output_tokens = (
    usage_df["output_tokens"]
    .fillna(0)
    .sum()
)


total_tokens = (
    usage_df["total_tokens"]
    .fillna(0)
    .sum()
)


if (
    "cost_usd" in usage_df.columns
    and
    usage_df["cost_usd"].notna().any()
):

    total_cost_usd = (
        usage_df["cost_usd"]
        .fillna(0)
        .sum()
    )

else:

    total_cost_usd = np.nan


# ============================================================
# Print summary
# ============================================================

print()

display(
    usage_df[
        [
            "stage",
            "rows",
            "input_tokens",
            "output_tokens",
            "total_tokens",
            "cost_usd",
            "elapsed_sec",
            "avg_latency_sec"
        ]
    ]
)


print()
print("=" * 70)
print("RQ2 FINAL RESOURCE SUMMARY")
print("=" * 70)


print(
    f"Expected API Calls : "
    f"{(main_exp + stab_exp) * 2}"
)


print(
    f"Input Tokens       : "
    f"{int(total_input_tokens):,}"
)


print(
    f"Output Tokens      : "
    f"{int(total_output_tokens):,}"
)


print(
    f"Total Tokens       : "
    f"{int(total_tokens):,}"
)


if not pd.isna(total_cost_usd):

    print(
        f"Total API Cost     : "
        f"${total_cost_usd:.6f}"
    )

else:

    print(
        "Total API Cost     : "
        "Not available "
        "(cost column not found)"
    )


print(
    f"Main Qwen Time     : "
    f"{main_qwen_sec / 60:.2f} min"
)


print(
    f"Main Llama Time    : "
    f"{main_llama_sec / 60:.2f} min"
)


print(
    f"Stability Qwen Time: "
    f"{stab_qwen_sec / 60:.2f} min"
)


print(
    f"Stability Llama Time: "
    f"{stab_llama_sec / 60:.2f} min"
)


print(
    f"TOTAL ELAPSED      : "
    f"{total_elapsed_sec / 60:.2f} min "
    f"({total_elapsed_sec / 3600:.2f} hours)"
)


print("=" * 70)


# ============================================================
# E. Save Usage Summary
# ============================================================

usage_summary_path = os.path.join(
    RQ2_OUTPUT_DIR,
    "rq2_api_usage_summary.csv"
)


atomic_csv_save(
    usage_df,
    usage_summary_path
)


overall_usage_df = pd.DataFrame([
    {
        "qwen_calls_expected":
            main_exp + stab_exp,

        "llama_calls_expected":
            main_exp + stab_exp,

        "total_api_calls_expected":
            (main_exp + stab_exp) * 2,

        "total_input_tokens":
            int(total_input_tokens),

        "total_output_tokens":
            int(total_output_tokens),

        "total_tokens":
            int(total_tokens),

        "total_cost_usd":
            total_cost_usd,

        "main_qwen_sec":
            main_qwen_sec,

        "main_llama_sec":
            main_llama_sec,

        "stability_qwen_sec":
            stab_qwen_sec,

        "stability_llama_sec":
            stab_llama_sec,

        "total_elapsed_sec":
            total_elapsed_sec,

        "total_elapsed_min":
            total_elapsed_sec / 60,

        "total_elapsed_hour":
            total_elapsed_sec / 3600
    }
])


overall_usage_path = os.path.join(
    RQ2_OUTPUT_DIR,
    "rq2_api_usage_total.csv"
)


atomic_csv_save(
    overall_usage_df,
    overall_usage_path
)


print()
print("✅ API usage summary saved:")
print(usage_summary_path)

print()
print("✅ Total resource summary saved:")
print(overall_usage_path)


# ------------------------------------------------------------
# Inspect token-related columns.
# ------------------------------------------------------------

print()
print("=" * 70)
print("TOKEN COLUMN CHECK")
print("=" * 70)

print(
    "Qwen columns:",
    main_narratives_df.columns.tolist()
)

print()

print(
    "Llama columns:",
    main_claims_df.columns.tolist()
)

In [ ]:
# ===== original cell 25 =====

print("\n" + "="*70)
print("RQ2 FINAL INTERPRETATION")
print("="*70)
print("ESR: Evidence Support Rate      → higher is better")
print("UCR: Unsupported Claim Rate    → lower is better")
print("CR : Contradiction Rate        → lower is better")
print("EC : Evidence Coverage         → higher is better")
print("\nCompare Baseline and HCLA. If HCLA")
print("  ① ESR↑  ② UCR↓  ③ CR↓  ④ EC↑")
print("this supports the effectiveness of evidence-grounded explanations.")
print("\nResult files:")
print(" -", os.path.join(RQ2_OUTPUT_DIR, "rq2_main50_paper_summary.csv"))
print(" -", os.path.join(RQ2_OUTPUT_DIR, "rq2_stability10x10_paper_summary.csv"))
print(" -", os.path.join(RQ2_OUTPUT_DIR, "rq2_stability10x10_by_rep.csv"))
print("="*70)
